# js.auto_fit

> JavaScript generator for automatic visible row count adjustment based on viewport height.

In [ ]:
#| default_exp js.auto_fit

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionUrls,
)

In [ ]:
#| export
def generate_auto_fit_js(
    ids: VirtualCollectionHtmlIds,       # HTML IDs for this collection
    config: VirtualCollectionConfig,      # Collection config (for row_height)
    urls: VirtualCollectionUrls,          # URL bundle (for update_viewport)
    debug: bool = False,                  # Enable console.log debugging
) -> str:  # JavaScript code fragment
    """Generate JS for auto-fitting visible row count to viewport height.
    
    Uses ResizeObserver on the viewport element to detect height changes from
    viewport-fit. Pure arithmetic: visibleRows = floor(viewportHeight / rowHeight).
    """
    if debug:
        dbg = """
                console.log('[auto-fit] called, viewport:', viewport?.id, 'offsetHeight:', viewportHeight, 'rowHeight:', rowHeight, 'visibleRows:', visibleRows, 'last:', _lastVisibleRows);
        """
        dbg_skip = """
                console.log('[auto-fit] skipped — same count:', visibleRows);
        """
        dbg_send = """
                console.log('[auto-fit] sending update_viewport:', visibleRows);
        """
    else:
        dbg = dbg_skip = dbg_send = ""

    return f"""
        // === Auto-Fit Visible Rows ===
        (function() {{
            let _lastVisibleRows = 0;

            window._vcAutoFit_{config.prefix} = function() {{
                const viewport = document.getElementById('{ids.viewport}');
                if (!viewport) return;

                const viewportHeight = viewport.offsetHeight;
                if (viewportHeight <= 0) return;

                const rowHeight = {config.row_height};
                const visibleRows = Math.max(1, Math.floor(viewportHeight / rowHeight));
                {dbg}
                // Only send update if count actually changed
                if (visibleRows === _lastVisibleRows) {{{dbg_skip}
                    return;
                }}
                _lastVisibleRows = visibleRows;
                {dbg_send}
                htmx.ajax('POST', '{urls.update_viewport}', {{
                    swap: 'none',
                    values: {{ visible_rows: visibleRows, is_auto: 'true' }}
                }});
            }};

            // Watch viewport for height changes (set by viewport-fit)
            const viewport = document.getElementById('{ids.viewport}');
            if (viewport) {{
                const observer = new ResizeObserver(function() {{
                    window._vcAutoFit_{config.prefix}();
                }});
                observer.observe(viewport);
            }}
        }})();
    """

In [ ]:
#| export
def auto_fit_callback_name(
    config: VirtualCollectionConfig,  # Collection config (for prefix)
) -> str:  # Global JS function name
    """Get the global callback name for viewport-fit's resize_callback."""
    return f"window._vcAutoFit_{config.prefix}()"

## Tests

In [ ]:
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds                                                        
from cjm_fasthtml_virtual_collection.core.models import VirtualCollectionConfig, VirtualCollectionUrls          

ids = VirtualCollectionHtmlIds(prefix="t")                                                     
config = VirtualCollectionConfig(prefix="t", row_height=40)
urls = VirtualCollectionUrls(update_viewport="/vc/update_viewport")                            
                              
js = generate_auto_fit_js(ids, config, urls)
assert 't-viewport' in js
assert '/vc/update_viewport' in js
assert '_vcAutoFit_t' in js
assert 'const rowHeight = 40' in js
assert 'Math.floor(viewportHeight / rowHeight)' in js
assert 'htmx.ajax' in js
assert '_lastVisibleRows' in js
print("Auto-fit JS generation test passed")

Auto-fit JS generation test passed


In [ ]:
# Test callback name
cb = auto_fit_callback_name(config)
assert cb == "window._vcAutoFit_t()"
print(f"Callback name: {cb}")

Callback name: window._vcAutoFit_t()


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()